# Visualise a Neo4j observation series and its candidates

This notebook creates a small synthetic observation-series dataset, loads the aircraft/radar knowledge graph and evidence graph into Neo4j, then inspects one `series_id` and its ranked `CandidateEvidence` nodes.

It demonstrates the `observation_series` input support in `rgcn_fusion.observation_etl.load_observations`: the series wrapper is flattened for ingestion while `series_id`, `sequence_index`, and `elapsed_time_s` are retained on Neo4j evidence nodes.

> **Prerequisite:** Start Neo4j 5 and install the repository dependencies. For example:
>
> ```bash
> docker run --rm --name rgcn-neo4j -p 7474:7474 -p 7687:7687 \
>   -e NEO4J_AUTH=neo4j/password neo4j:5
> ```
>
> This demonstration clears the selected database before loading its data. Set `RESET_DATABASE = False` if the database contains data you need to keep.


In [ ]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "kg_generator.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = 'password123'#os.getenv("NEO4J_PASSWORD", "password")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
RESET_DATABASE = True
GENERATED = REPO_ROOT / "generated"
GENERATED.mkdir(exist_ok=True)


## 1. Generate a compact observation-series document

Each wrapper entry represents one emitter track. The ETL accepts this wrapper directly; no manual conversion to an `observations` list is required.


In [ ]:
from datetime import UTC, datetime
from esm_observation_series_generator import generate_observation_series

series_document = generate_observation_series(
    count=2,
    seed=7,
    min_duration_s=4.0,
    max_duration_s=4.0,
    sample_interval_s=0.5,
    mode_switch_probability=0.35,
    workers=1,
    start=datetime(2025, 1, 1, tzinfo=UTC),
    end=datetime(2025, 1, 2, tzinfo=UTC),
)
series_path = GENERATED / "esm_observation_series_visualisation_demo.json"
series_path.write_text(json.dumps(series_document, indent=2), encoding="utf-8")

[(entry["series_id"], entry["observation_count"]) for entry in series_document["observation_series"]]


## 1b. Load existing synthetic observations

To inspect an existing synthetic input instead of generating new data, use `load_observations` with its path. It accepts both a flat `observations` document and an `observation_series` wrapper. The checked-in flat sample is loaded below without changing the generated series used by the remaining visualisation cells.

For a pre-existing **series** document, set `series_path` to that file before running the ingestion cell. Its `series_id` and ordering fields will then be available to the Neo4j graph and timeline queries.


In [ ]:
from rgcn_fusion.observation_etl import load_observations

existing_observations_path = GENERATED / "esm_observations.json"
existing_observations = load_observations(existing_observations_path)
print(f"Loaded {len(existing_observations)} existing synthetic observations from {existing_observations_path}")
existing_observations[0]


## 2. Load the base knowledge graph

Candidate scoring queries `RadarMode`, `Radar`, `AircraftVariant`, and `Operator` nodes from Neo4j, so load the base graph before ingesting observations.


In [ ]:
from neo4j import GraphDatabase
from kg_generator import generate_graph


driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
graph = generate_graph()


def clear_database(tx):
    tx.run("MATCH (n) DETACH DELETE n")


def load_nodes(tx, label, rows):
    tx.run(
        f"""
        UNWIND $rows AS row
        MERGE (node:{label} {{id: row.id}})
        SET node += row.properties
        SET node.id = row.id, node.label = row.label
        """,
        rows=rows,
    )


def load_relationships(tx, relation, rows):
    tx.run(
        f"""
        UNWIND $rows AS row
        MATCH (source {{id: row.source}})
        MATCH (target {{id: row.target}})
        MERGE (source)-[:{relation}]->(target)
        """,
        rows=rows,
    )


with driver.session(database=NEO4J_DATABASE) as session:
    if RESET_DATABASE:
        session.execute_write(clear_database)
    for label in sorted({node["label"] for node in graph["nodes"]}):
        rows = [node for node in graph["nodes"] if node["label"] == label]
        session.execute_write(load_nodes, label, rows)
    for relation in sorted({edge["relation"] for edge in graph["edges"]}):
        rows = [edge for edge in graph["edges"] if edge["relation"] == relation]
        session.execute_write(load_relationships, relation, rows)

print(f"Loaded {len(graph['nodes'])} nodes and {len(graph['edges'])} relationships.")


## 3. Ingest the series wrapper as evidence nodes

`load_observations` flattens `observation_series[*].observations`. The following assertion confirms that a child observation inherited its parent `series_id` if necessary, and that its ordering fields remain available to the ETL.


In [ ]:
from rgcn_fusion.observation_etl import ObservationNeo4jETL, load_observations

observations = load_observations(series_path)
assert all("series_id" in observation for observation in observations)
assert all("sequence_index" in observation for observation in observations)

etl = ObservationNeo4jETL(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
try:
    ingest_result = etl.ingest(observations, max_candidates=5)
finally:
    etl.close()
ingest_result


## 4. Select a series and inspect its observation count

The returned rows are the values to use for the `$seriesId` Cypher parameter. The query also confirms that the persisted track properties can be used to order the series.


In [ ]:
available_series_query = """
MATCH (observation:Observation)
WHERE observation.series_id IS NOT NULL
RETURN observation.series_id AS series_id,
       count(*) AS observation_count,
       min(observation.timestamp_iso8601) AS first_seen,
       max(observation.timestamp_iso8601) AS last_seen
ORDER BY first_seen, series_id
"""

with driver.session(database=NEO4J_DATABASE) as session:
    available_series = session.run(available_series_query).data()
available_series


In [ ]:
series_id = available_series[0]["series_id"]
print(f"Visualising {series_id}")


## 5. Render the subgraph in Neo4j Browser or Workspace

The Neo4j Python driver returns graph entities but does not render the Browser graph canvas. To use Neo4j's interactive graph view, open `http://localhost:7474`, set the parameters below, and run the graph query. It includes each observation's top three candidates and any outgoing contradiction edges between candidates in the selected series.


In [ ]:
# Paste these commands into Neo4j Browser / Workspace graph view.
# :param seriesId => 'replace-with-the-series-id-printed-above';
# :param candidateLimit => 3;
#
# MATCH (observation:Observation {series_id: $seriesId})
# CALL {
#   WITH observation
#   MATCH (observation)-[has_candidate:HAS_CANDIDATE]->(candidate:CandidateEvidence)
#   WITH candidate, has_candidate
#   ORDER BY has_candidate.rank ASC
#   LIMIT $candidateLimit
#   RETURN collect({candidate: candidate, relationship: has_candidate}) AS candidate_rows
# }
# UNWIND candidate_rows AS candidate_row
# WITH observation, candidate_row.candidate AS candidate,
#      candidate_row.relationship AS has_candidate
# OPTIONAL MATCH (candidate)-[contradiction:CONTRADICTS_CANDIDATE]->(other:CandidateEvidence)
# WHERE other.series_id = $seriesId
# RETURN observation, has_candidate, candidate, contradiction, other
# ORDER BY observation.sequence_index, has_candidate.rank;


For readability, use `sequence_index` as the `Observation` caption, `rank` as the `CandidateEvidence` caption, and colour `CONTRADICTS_CANDIDATE` relationships red. See `docs/neo4j_observation_series_visualisation.md` for the same query and styling guidance.

## 6. Inspect the ranked candidate timeline in this notebook

The tabular query is useful for long tracks and makes mode changes or candidate-score drops easy to spot. Rank one is the best candidate for each observation.


In [ ]:
timeline_query = """
MATCH (observation:Observation {series_id: $series_id})
MATCH (observation)-[has_candidate:HAS_CANDIDATE {rank: 1}]->(candidate:CandidateEvidence)
RETURN observation.sequence_index AS sequence,
       observation.elapsed_time_s AS elapsed_seconds,
       observation.timestamp_iso8601 AS timestamp,
       candidate.mode_id AS mode,
       candidate.radar_id AS radar,
       candidate.aircraft_id AS aircraft,
       candidate.operator AS operator,
       has_candidate.score AS score
ORDER BY sequence
"""

with driver.session(database=NEO4J_DATABASE) as session:
    timeline = session.run(timeline_query, series_id=series_id).data()
timeline


## 7. Clean up the driver connection

The data remains in Neo4j for interactive Browser exploration after this notebook finishes.


In [ ]:
driver.close()
